# Your first analysis: a two-component dataset 🔬

This notebook runs a complete **global analysis** from start to finish
on a small, clean dataset that comes from a **CSV file** — the same
kind of file most instruments can export and that you can open in
Excel.

*Global* means we analyse **all wavelengths at once** with a single
model, instead of fitting one trace at a time. The data describe a
sample that relaxes through **two states, one after another**
(s1 → s2 → ground), measured over time and across wavelengths.

Global and target analysis follows a small **cycle** that you will
repeat in every notebook and in your own work:

1. **Inspect** the data — see how the signal evolves in time and
   across wavelength, and get a feel for how many states you will need.
2. **Specify** a model: a kinetic scheme (what we think is happening)
   plus starting **parameters** (our initial guesses).
3. **Estimate** the parameters — let the optimizer refine them so the
   model reproduces the data as closely as possible.
4. **Validate and interpret** — check that the fit really describes the
   data, then read off the states' spectra and lifetimes.

In real work you go around this loop several times, refining the model.
Here the model is already right, so one pass is enough. (Saving the
result at the end is optional — useful for reproducibility, but not a
step of the analysis itself.)

Run each cell with **Shift + Enter**, in order.

## Step 0: Load the tools we need

`load_csv_dataset` is a small helper that ships with this kit
(`csv_tools.py`, right next to this notebook). pyglotaran does not read
plain CSV files on its own *yet*, so we use this helper to turn a CSV
export into data pyglotaran understands. Feel free to open
`csv_tools.py` and look — it is only a few lines.

In [ ]:
from csv_tools import load_csv_dataset

from glotaran.io import save_result
from glotaran.optimization.optimize import optimize
from glotaran.project.scheme import Scheme

from pyglotaran_extras import plot_data_overview, plot_overview

## Step 1: Inspect the data

The measurement lives in `02_two_component/data.csv`. Open it from the
file browser on the left if you are curious: the first column is time,
the first row is wavelength, and every other cell is a measured value.

The plot below is a standard *data overview*. The large panel is the
measured signal as a function of time and wavelength; the smaller
panels summarise its main features — how the spectrum **evolves** over
time and how individual traces **decay**. A quick look tells you
roughly how many states the model will need. Always inspect the data
this way before specifying a model.

In [ ]:
data = load_csv_dataset("02_two_component/data.csv")

plot_data_overview(data, linlog=True, linthresh=1);

## Step 2: Specify the model and starting parameters

Two small text files describe the analysis. You can open them from the
file browser to peek inside:

- `02_two_component/model.yaml` — the **model**: a **kinetic scheme** of
  two states decaying in sequence (a *compartmental* model), plus a
  Gaussian instrument response function (IRF) that accounts for the
  finite time resolution of the measurement.
- `02_two_component/parameters.yaml` — the **starting guesses** for the
  decay rates and the IRF.

We bundle them together with the data into a **Scheme** — think of it as
the complete recipe for one analysis. `validate()` checks the recipe
makes sense before we run it.

In [ ]:
scheme = Scheme(
    model="02_two_component/model.yaml",
    parameters="02_two_component/parameters.yaml",
    data={"dataset1": data},
    maximum_number_function_evaluations=99,
)

scheme.validate()

If the line above says the model is valid, you are ready to run the
analysis. If it lists problems, they are usually a typo in one of the
YAML files.

## Step 3: Estimate the parameters (optimize)

This is where the computer does the work: it adjusts the parameters
step by step — a nonlinear least-squares fit — until the model matches
the data as closely as possible.

It takes a few seconds. While it runs you will see `[*]` next to the
cell. When it finishes, a summary table appears.

In [ ]:
result = optimize(scheme)
result

The table above summarises how well the fit worked. Next, let's see
the refined parameters — the two decay rates the analysis found.

In [ ]:
result.optimized_parameters

## Step 4: Validate and interpret the results

The overview plot below is the payoff, and it does two jobs at once.

**Validate** — the panels comparing the model with the measured data,
and the residuals, tell you whether the fit actually describes the
data. Structure left in the residuals would mean the model is missing
something.

**Interpret** — once you trust the fit, you read off the science: the
**concentration profiles** (how much of each state is present over
time) and one spectrum per state. Because we used a **sequential**
scheme with increasing lifetimes, these are **evolution-associated
spectra**: they trace how the overall spectrum evolves as the system
moves down the s1 → s2 chain.

In [ ]:
plot_overview(result.data["dataset1"], linlog=True, linthresh=10);

## (Optional) Save the results

Saving is **not part of the analysis loop**, but it makes your work
reproducible — a core aim of pyglotaran and of FAIR data. This writes
everything (fitted parameters, curves, and plots) into a new
`results/` folder next to this notebook, so you can come back to it
later or open it in other tools.

In [ ]:
save_result(
    result=result,
    result_path="02_two_component/results/result.yml",
    allow_overwrite=True,
)
print("Saved! Look in the 02_two_component/results folder on the left.")

## 🎉 You did it

You just ran a complete global analysis from a data file: you inspected
the data, specified a model, estimated its parameters, and validated
and interpreted the result.

### What next?

Open **`03_three_component.ipynb`** to run the same cycle on a slightly
richer dataset — one with **three** states instead of two. The workflow
is identical; only the model grows by one state.

### Want to experiment?

Try changing a starting value in `02_two_component/parameters.yaml`
(for example one of the `kinetic` numbers), save the file, then run
this notebook again from the top with **Kernel → Restart Kernel and Run
All Cells**. Seeing how the fit responds is one of the best ways to
build intuition.